In [ ]:
# 1. Environment & Path Setup
import os
import sys

# Add the project root directory to sys.path so 'pipeline' is importable
sys.path.append(os.path.abspath('../..'))

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler
from matplotlib.lines import Line2D
from mplsoccer import Pitch
from sklearn.cluster import KMeans


In [ ]:
QUERY = """
-- primary table
CREATE TEMP TABLE pass_distance AS SELECT
    player_id,
    first(player) AS player,
    first(team_id) AS team_id,
    first(team) AS team,
    avg(pass_end_x - x) AS average_pass_x_displacement,
    avg(abs(pass_end_y - y)) AS average_pass_norm_y_displacement,
    count(*) AS pass_total
FROM events
WHERE
    (
        (type = 'Pass') AND
        (position <> 'Goalkeeper')
    ) 
    AND (mandown = False)
    AND
    (
        (play_pattern = 'Regular Play') OR
        (play_pattern = 'From Keeper') OR
        (play_pattern = 'From Counter') OR
        (play_pattern = 'From Throw In')
    )
    AND (matchweek >= $1 AND matchweek <= $2)
GROUP BY player_id;

-- primary table
CREATE TEMP TABLE carry_distance AS SELECT
    player_id,
    first(player) AS player,
    avg(end_x - x) AS average_carry_x_displacement,
    avg(abs(end_y - y)) AS average_carry_norm_y_displacement,
    count(*) AS carry_total
FROM events
WHERE
    (
        (type = 'Complete Carry')
    ) 
    AND (mandown = False)
    AND
    (
        (play_pattern = 'Regular Play') OR
        (play_pattern = 'From Keeper') OR
        (play_pattern = 'From Counter') OR
        (play_pattern = 'From Throw In')
    )
    AND (matchweek >= $1 AND matchweek <= $2)
GROUP BY player_id;

-- primary table
CREATE TEMP TABLE player_rel AS SELECT
    player_id,
    match_id,
    team_id,
    player,
    position,
    type,
    match_seconds,
    teamsheet,
    mandown,
    play_pattern,
    CASE
        WHEN type = 'Complete Carry' THEN end_x
        ELSE x
        END AS x,
    CASE
        WHEN type = 'Complete Carry' THEN end_y
        ELSE y
        END AS y
FROM
    events
WHERE
    (
        (type = 'Pass') OR
        (type = 'Shot') OR
        ((type = 'Complete Carry') AND (carry_end_outcome = 'Other'))
    ) 
    AND (mandown = False)
    AND
    (
        (play_pattern = 'Regular Play') OR
        (play_pattern = 'From Keeper') OR
        (play_pattern = 'From Counter') OR
        (play_pattern = 'From Throw In')
    )
    AND (matchweek >= $1 AND matchweek <= $2);

-- secondary table from "player_rel"
CREATE TEMP TABLE player_rel_avg AS SELECT
    player_id,
    first(player) AS player,
    avg(x) AS avg_rel_x,
    avg(y) AS avg_rel_y,
    avg(40 - ABS(y - 40)) AS avg_normrel_y,
    count(*) AS rel_total
FROM player_rel
GROUP BY player_id;

-- primary table
CREATE TEMP TABLE player_rec AS SELECT
    player_id,
    match_id,
    team_id,
    player,
    position,
    type,
    match_seconds,
    teamsheet,
    mandown,
    play_pattern,
    x,
    y,
FROM
    events
WHERE 
    (
        (type = 'Ball Receipt*') OR
        (type = 'Ball Recovery') OR
        ((type = '50/50') AND ("50_50".outcome.name = 'Won')) OR
        ((type = 'Duel') AND (duel_outcome = 'Won')) OR
        ((type = 'Interception') AND (interception_outcome = 'Won')) OR
        ((type = 'Pass') AND (pass_type = 'Recovery')) OR
        ((type = 'Shot') AND (shot_type = 'Recovery'))
    )
    AND (mandown = False)
    AND
    (
        (play_pattern = 'Regular Play') OR
        (play_pattern = 'From Keeper') OR
        (play_pattern = 'From Counter') OR
        (play_pattern = 'From Throw In')
    )
    AND (matchweek >= $1 AND matchweek <= $2);

-- secondary table from "player_rec"
CREATE TEMP TABLE player_rec_avg AS SELECT
    player_id,
    first(player) AS player,
    avg(x) AS avg_rec_x,
    avg(y) AS avg_rec_y,
    avg(40 - ABS(y - 40)) AS avg_normrec_y,
    count(*) AS rec_total
FROM player_rec
GROUP BY player_id;

-- primary table
CREATE TEMP TABLE key_pass AS SELECT
    player_id,
    first(player) AS player,
    count(*) AS key_pass_total
FROM
    events
WHERE
    (
        (type = 'Pass') AND
        (pass_shot_assist)
    )
    AND (matchweek >= $1 AND matchweek <= $2)
GROUP BY player_id;

-- primary table
CREATE TEMP TABLE shot_count AS SELECT
    player_id,
    first(player) AS player,
    count(*) AS shot_total
FROM events
WHERE type = 'Shot' AND (matchweek >= $1 AND matchweek <= $2)
GROUP BY player_id;

-- primary table
CREATE TEMP TABLE reception AS
    SELECT
        player_id,
        match_id,
        team_id,
        player,
        type,
        match_seconds,
        play_pattern,
        x,
        y,
    FROM
        events
    WHERE
        (
            (type = 'Ball Receipt*') OR
            (type = 'Ball Recovery') OR
            ((type = '50/50') AND ("50_50".outcome.name = 'Won')) OR
            ((type = 'Duel') AND (duel_outcome = 'Won')) OR
            ((type = 'Interception') AND (interception_outcome = 'Won'))
        )
        AND (matchweek >= $1 AND matchweek <= $2)
    ORDER BY
        match_id, team_id, match_seconds;

-- primary table
CREATE TEMP TABLE outcome AS
    SELECT
        player_id,
        match_id,
        team_id,
        type,
        match_seconds
    FROM
        events
    WHERE
        (
            (type = 'Pass') OR
            (type = 'Shot') OR
            (type = 'Complete Carry')
        )
        AND (matchweek >= $1 AND matchweek <= $2)
    ORDER BY
        match_id, team_id, match_seconds;
        -- Total shots taken while both were co-present on pitch
-- secondary table from "reception" and "outcome"
CREATE TEMP TABLE reception_outcomes AS
SELECT
    l.player_id as player_id,
    l.match_id as match_id,
    l.team_id as team_id,
    l.player as player,
    l.type as type,
    l.match_seconds as match_seconds,
    l.play_pattern as play_pattern,
    l.x as x,
    l.y as y,
    r.type as reception_outcome
FROM reception l LEFT JOIN outcome r ON
    l.player_id = r.player_id AND
    l.match_id = r.match_id AND
    l.team_id = r.team_id AND
    l.match_seconds = r.match_seconds;

-- tertiary table from "reception_outcomes"
CREATE TEMP TABLE outcome_pass_count AS
SELECT
    player_id,
    first(player) as player,
    count(*) as outcome_count
FROM reception_outcomes
WHERE reception_outcome = 'Pass'
GROUP BY player_id;

-- tertiary table from "reception_outcomes"
CREATE TEMP TABLE outcome_carry_count AS
SELECT
    player_id,
    first(player) as player,
    count(*) as outcome_count
FROM reception_outcomes
WHERE reception_outcome = 'Complete Carry'
GROUP BY player_id;

-- tertiary table from "reception_outcomes"
CREATE TEMP TABLE outcome_shot_count AS
SELECT
    player_id,
    first(player) as player,
    count(*) as outcome_count
FROM reception_outcomes
WHERE reception_outcome = 'Shot'
GROUP BY player_id;

-- returned table
SELECT
    l.player_id AS player_id,
    l.player AS player,
    l.team_id AS team_id,
    l.team AS team,
    l.average_pass_x_displacement AS average_pass_x_displacement,
    l.average_pass_norm_y_displacement AS average_pass_norm_y_displacement,
    l.pass_total AS pass_total,
    r.average_carry_x_displacement AS average_carry_x_displacement,
    r.average_carry_norm_y_displacement AS average_carry_norm_y_displacement,
    r.carry_total AS carry_total,
    a.avg_rel_x AS average_release_x,
    a.avg_rel_y AS average_release_y,
    a.avg_normrel_y AS average_norm_release_y,
    a.rel_total AS rel_total,
    b.avg_rec_x AS average_reception_x,
    b.avg_rec_y AS average_reception_y,
    b.avg_normrec_y AS average_norm_reception_y,
    b.rec_total AS rec_total,
    (c.key_pass_total / (c.key_pass_total + d.shot_total)) AS key_pass_shot_ratio,
    c.key_pass_total AS key_pass_total,
    d.shot_total AS shot_total,
    (oc.outcome_count/(op.outcome_count + oc.outcome_count + osh.outcome_count)) AS carry_shotpass_ratio,
    op.outcome_count AS outcome_pass_total,
    oc.outcome_count AS outcome_carry_total,
    osh.outcome_count AS outcome_shot_total,
FROM pass_distance l
INNER JOIN carry_distance r ON
    l.player_id = r.player_id AND
    l.player = r.player
INNER JOIN player_rel_avg a ON
    l.player_id = a.player_id AND
    l.player = a.player
INNER JOIN player_rec_avg b ON
    l.player_id = b.player_id AND
    l.player = b.player
INNER JOIN key_pass c ON
    l.player_id = c.player_id AND
    l.player = c.player
INNER JOIN shot_count d ON
    l.player_id = d.player_id AND
    l.player = d.player
INNER JOIN outcome_pass_count op ON
    l.player_id = op.player_id AND
    l.player = op.player
INNER JOIN outcome_carry_count oc ON
    l.player_id = oc.player_id AND
    l.player = oc.player
INNER JOIN outcome_shot_count osh ON
    l.player_id = osh.player_id AND
    l.player = osh.player
WHERE
    (a.rel_total >= $1 AND
    b.rec_total >= $1);
"""

In [ ]:
statements = [s.strip() for s in QUERY.split(';') if s.strip()]

setup_statements = statements[:-1]
final_query = statements[-1]

PRIMARY_TABLES = [0,1,2,4,6,7,8,9]
SECONDARY_TABLES = [3,5,10]
TERTIARY_TABLES = [11,12,13]

DB_PATH = '../../data/events.db'
con = duckdb.connect(DB_PATH, read_only = True)
for x in PRIMARY_TABLES:
    con.execute(setup_statements[x], [0,40])
for y in SECONDARY_TABLES:
    con.execute(setup_statements[y])
for z in TERTIARY_TABLES:
    con.execute(setup_statements[z])
main_df = con.execute(final_query, [50]).df()
con.close()

In [ ]:
# Ward-linkage clustering metrics using the requested feature columns
feature_cols = [
    "average_pass_x_displacement",
    "average_pass_norm_y_displacement",
    "average_carry_x_displacement",
    "average_carry_norm_y_displacement",
    "average_release_x",
    "average_norm_release_y",
    "average_reception_x",
    "average_norm_reception_y",
    "key_pass_shot_ratio",
    "carry_shotpass_ratio",
]
K_RANGE = list(range(2, 31))
RANDOM_STATE = 42
SILHOUETTE_SAMPLE = None  # Set to e.g. 10_000 if silhouette calculation is too slow.

cluster_df = main_df.dropna(subset=feature_cols).copy()
X = StandardScaler().fit_transform(cluster_df[feature_cols])

# Ward linkage uses Euclidean distance on the scaled feature matrix.
Z = linkage(X, method='ward')

rows = []
for k in K_RANGE:
    labels = fcluster(Z, t=k, criterion='maxclust')
    wss = sum(
        ((X[labels == cluster] - X[labels == cluster].mean(axis=0)) ** 2).sum()
        for cluster in np.unique(labels)
    )
    rows.append({
        'k': k,
        'wss': wss,
        'silhouette': (
            silhouette_score(
                X, labels, sample_size=SILHOUETTE_SAMPLE, random_state=RANDOM_STATE
            ) if k > 1 else np.nan
        ),
        'calinski_harabasz': calinski_harabasz_score(X, labels) if k > 1 else np.nan,
        'davies_bouldin': davies_bouldin_score(X, labels) if k > 1 else np.nan,
        'merge_height': Z[-(k - 1), 2] if k > 1 else np.nan,
    })

metrics_df = pd.DataFrame(rows).set_index('k')
print(metrics_df.round(3))

# Dendrogram
fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(
    Z, truncate_mode='lastp', p=30, show_leaf_counts=True,
    leaf_rotation=90, leaf_font_size=9, ax=ax,
)
ax.set_title('Ward linkage dendrogram (truncated to last 30 merges)')
ax.set_xlabel('Cluster size (or index)')
ax.set_ylabel('Ward distance')
plt.tight_layout()
plt.show()

# Validation metrics per cluster count
fig, axes = plt.subplots(5, 1, figsize=(14, 20))
panels = [
    ('wss', 'Within-cluster SS (elbow)', 'tab:blue'),
    ('silhouette', 'Silhouette (higher better)', 'tab:orange'),
    ('calinski_harabasz', 'Calinski-Harabasz (higher better)', 'tab:green'),
    ('davies_bouldin', 'Davies-Bouldin (lower better)', 'tab:red'),
    ('merge_height', 'Merge height (big jump = natural cut)', 'tab:purple'),
]
for ax, (column, title, color) in zip(axes, panels):
    ax.plot(metrics_df.index, metrics_df[column], marker='o', color=color)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Number of clusters (k)')
    ax.set_xlim(2, 30)
    ax.set_xticks(np.arange(2, 31, 1))
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Ward-linkage pitch plot for k = 3
k = 3
position_plot_df = cluster_df.copy()
position_plot_df['cluster'] = fcluster(Z, t=k, criterion='maxclust')
cluster_colors = plt.cm.tab20.colors
pitch = Pitch(pitch_type='statsbomb', pitch_color='white', line_color='0.6', linewidth=1)
fig, ax = pitch.draw(figsize=(12, 8))

for cluster, group in position_plot_df.groupby('cluster'):
    color = cluster_colors[(cluster - 1) % len(cluster_colors)]
    # Plot raw average release and reception coordinates; scaled features are only used for clustering.
    pitch.lines(
        group['average_reception_x'], group['average_reception_y'],
        group['average_release_x'], group['average_release_y'],
        ax=ax, color=color, lw=0.8, alpha=0.25, zorder=2,
    )
    pitch.scatter(
        group['average_reception_x'], group['average_reception_y'],
        ax=ax, marker='o', s=18, color=color, alpha=0.65, zorder=3,
    )
    pitch.scatter(
        group['average_release_x'], group['average_release_y'],
        ax=ax, marker='s', s=18, color=color, alpha=0.65, zorder=3,
    )

ax.set_title(f'Ward linkage clusters (k = {k}) | n = {len(position_plot_df)}')
ax.legend(
    handles=[
        Line2D([], [], color=cluster_colors[(cluster - 1) % len(cluster_colors)],
               linewidth=2, label=f'Cluster {cluster}')
        for cluster in sorted(position_plot_df['cluster'].unique())
    ],
    title='Cluster', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False,
)
plt.tight_layout()
plt.show()


In [ ]:
# Ward-linkage pitch plot for k = 5
k = 5
position_plot_df = cluster_df.copy()
position_plot_df['cluster'] = fcluster(Z, t=k, criterion='maxclust')
cluster_colors = plt.cm.tab20.colors
pitch = Pitch(pitch_type='statsbomb', pitch_color='white', line_color='0.6', linewidth=1)
fig, ax = pitch.draw(figsize=(12, 8))

for cluster, group in position_plot_df.groupby('cluster'):
    color = cluster_colors[(cluster - 1) % len(cluster_colors)]
    # Plot raw average release and reception coordinates; scaled features are only used for clustering.
    pitch.lines(
        group['average_reception_x'], group['average_reception_y'],
        group['average_release_x'], group['average_release_y'],
        ax=ax, color=color, lw=0.8, alpha=0.25, zorder=2,
    )
    pitch.scatter(
        group['average_reception_x'], group['average_reception_y'],
        ax=ax, marker='o', s=18, color=color, alpha=0.65, zorder=3,
    )
    pitch.scatter(
        group['average_release_x'], group['average_release_y'],
        ax=ax, marker='s', s=18, color=color, alpha=0.65, zorder=3,
    )

ax.set_title(f'Ward linkage clusters (k = {k}) | n = {len(position_plot_df)}')
ax.legend(
    handles=[
        Line2D([], [], color=cluster_colors[(cluster - 1) % len(cluster_colors)],
               linewidth=2, label=f'Cluster {cluster}')
        for cluster in sorted(position_plot_df['cluster'].unique())
    ],
    title='Cluster', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False,
)
plt.tight_layout()
plt.show()


In [ ]:
# Ward-linkage pitch plot for k = 9
k = 9
position_plot_df = cluster_df.copy()
position_plot_df['cluster'] = fcluster(Z, t=k, criterion='maxclust')
cluster_colors = plt.cm.tab20.colors
pitch = Pitch(pitch_type='statsbomb', pitch_color='white', line_color='0.6', linewidth=1)
fig, ax = pitch.draw(figsize=(12, 8))

for cluster, group in position_plot_df.groupby('cluster'):
    color = cluster_colors[(cluster - 1) % len(cluster_colors)]
    # Plot raw average release and reception coordinates; scaled features are only used for clustering.
    pitch.lines(
        group['average_reception_x'], group['average_reception_y'],
        group['average_release_x'], group['average_release_y'],
        ax=ax, color=color, lw=0.8, alpha=0.25, zorder=2,
    )
    pitch.scatter(
        group['average_reception_x'], group['average_reception_y'],
        ax=ax, marker='o', s=18, color=color, alpha=0.65, zorder=3,
    )
    pitch.scatter(
        group['average_release_x'], group['average_release_y'],
        ax=ax, marker='s', s=18, color=color, alpha=0.65, zorder=3,
    )

ax.set_title(f'Ward linkage clusters (k = {k}) | n = {len(position_plot_df)}')
ax.legend(
    handles=[
        Line2D([], [], color=cluster_colors[(cluster - 1) % len(cluster_colors)],
               linewidth=2, label=f'Cluster {cluster}')
        for cluster in sorted(position_plot_df['cluster'].unique())
    ],
    title='Cluster', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False,
)
plt.tight_layout()
plt.show()


In [ ]:
# Ward-linkage pitch plot for k = 17
k = 17
position_plot_df = cluster_df.copy()
position_plot_df['cluster'] = fcluster(Z, t=k, criterion='maxclust')
cluster_colors = plt.cm.tab20.colors
pitch = Pitch(pitch_type='statsbomb', pitch_color='white', line_color='0.6', linewidth=1)
fig, ax = pitch.draw(figsize=(12, 8))

for cluster, group in position_plot_df.groupby('cluster'):
    color = cluster_colors[(cluster - 1) % len(cluster_colors)]
    # Plot raw average release and reception coordinates; scaled features are only used for clustering.
    pitch.lines(
        group['average_reception_x'], group['average_reception_y'],
        group['average_release_x'], group['average_release_y'],
        ax=ax, color=color, lw=0.8, alpha=0.25, zorder=2,
    )
    pitch.scatter(
        group['average_reception_x'], group['average_reception_y'],
        ax=ax, marker='o', s=18, color=color, alpha=0.65, zorder=3,
    )
    pitch.scatter(
        group['average_release_x'], group['average_release_y'],
        ax=ax, marker='s', s=18, color=color, alpha=0.65, zorder=3,
    )

ax.set_title(f'Ward linkage clusters (k = {k}) | n = {len(position_plot_df)}')
ax.legend(
    handles=[
        Line2D([], [], color=cluster_colors[(cluster - 1) % len(cluster_colors)],
               linewidth=2, label=f'Cluster {cluster}')
        for cluster in sorted(position_plot_df['cluster'].unique())
    ],
    title='Cluster', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False,
)
plt.tight_layout()
plt.show()


In [ ]:
# K-means elbow and silhouette evaluation for k=2..20
KMEANS_K_RANGE = list(range(2, 21))
kmeans_cluster_df = main_df.dropna(subset=feature_cols).copy()
kmeans_X = StandardScaler().fit_transform(kmeans_cluster_df[feature_cols])

kmeans_inertias = []
kmeans_silhouettes = []
for k in KMEANS_K_RANGE:
    kmeans_model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    kmeans_labels = kmeans_model.fit_predict(kmeans_X)
    kmeans_inertias.append(kmeans_model.inertia_)
    kmeans_silhouettes.append(
        silhouette_score(
            kmeans_X,
            kmeans_labels,
            sample_size=min(10_000, len(kmeans_cluster_df)),
            random_state=RANDOM_STATE,
        )
    )

kmeans_metrics_df = pd.DataFrame(
    {'elbow_inertia': kmeans_inertias, 'silhouette': kmeans_silhouettes},
    index=pd.Index(KMEANS_K_RANGE, name='k'),
)
print(kmeans_metrics_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(KMEANS_K_RANGE, kmeans_inertias, marker='o')
axes[0].set_title('K-means elbow method')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Inertia (within-cluster sum of squares)')
axes[0].set_xticks(KMEANS_K_RANGE)

axes[1].plot(KMEANS_K_RANGE, kmeans_silhouettes, marker='o', color='tab:orange')
axes[1].set_title('K-means silhouette score')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Mean silhouette score')
axes[1].set_xticks(KMEANS_K_RANGE)

for ax in axes:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()
